In [1]:
import re
import requests, time
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
PATH_TO_PLAYERS_LINK= "All_Players_Link.csv"
PATH_TO_PLAYERS_DATA= "Players_Data.csv"
PATH_TO_NOT_GK_PLAYERS_DATA= "Not_GK_Players_Data.csv"
PATH_TO_GK_PLAYERS_DATA= "GK_Players_Data.csv"
PATH_TO_NOT_GK_PLAYERS_LINK= "Not_GK_Players_Link.csv"
PATH_TO_GK_PLAYERS_LINK= "GK_Players_Link.csv"
PATH_TO_NOT_GK_PLAYERS_STAT= "Not_GK_Players_Stat.csv"
PATH_TO_GK_PLAYERS_STAT= "GK_Players_Stat.csv"
PATH_TO_NOT_GK_PLAYERS= "Not_GK_Players.csv"
PATH_TO_GK_PLAYERS= "GK_Players_Data.csv"
PATH_TO_FINAL_DATA= "Final_Data.csv"

In [3]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36", "accept-language": "en-US,en;q=0.9"}

In [4]:
# country = pd.DataFrame([
#         {'CountryID': 189, 'Country': 'England'},
#         {'CountryID': 40, 'Country': 'Germany'},
#         {'CountryID': 75, 'Country': 'Italy'},
#         {'CountryID': 50, 'Country': 'France'},
#         {'CountryID': 157, 'Country': 'Spain'}
#         ])

In [6]:
# league_name, league_url = [], []
# for i in range(len(country)):
#     url = f"https://www.transfermarkt.com/wettbewerbe/national/wettbewerbe/{country.loc[i,'CountryID']}"
#     time.sleep(1)
#     page = requests.get(url, headers = headers)
#     soup = BeautifulSoup(page.content, 'html.parser')
    
#     for j in range(1, 3):
#         league_span = soup.select('.inline-table a')[j]
#         league_name.append(league_span.get('title'))
#         league_url.append('https://www.transfermarkt.com' + league_span.get('href') + '/plus/?saison_id=')

In [7]:
# All_Players_Link = []
# for league, url in zip(league_name, league_url):
#     for season in range(2024, 2025):
#         page = requests.get(url + str(season), headers = headers)
#         soup = BeautifulSoup(page.content, 'html.parser')
        
#         club_urls = [link.get("href") for link in soup.select("#yw1 .no-border-links a:nth-child(1)")]
#         for club_url in club_urls:
#             club_id = club_url.split("/")[-3]
#             club_page = requests.get("https://www.transfermarkt.com" + club_url, headers=headers)    
#             club_link = "https://www.transfermarkt.com" + club_url
#             print(f"Scraping {club_link} for players...")           
#             soup2 = BeautifulSoup(club_page.content, "html.parser")

#             players_list = soup2.select(".inline-table .hauptlink > a")
#             All_Players_Link.extend(p.get('href') for p in players_list)
#             time.sleep(2)

In [8]:
# PATH_TO_PLAYERS_LINK = 'All_Players_Link.csv'
# pd.DataFrame(All_Players_Link).drop_duplicates().to_csv(PATH_TO_PLAYERS_LINK, index = False)

In [9]:
def scraping_players_data(url, players_datas):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36", "accept-language": "en-US,en;q=0.9"}
    page = requests.get(url, headers=headers)
    # time.sleep(3)
    soup = BeautifulSoup(page.content, "html.parser")
    data = {}

    pattern = r"/(\d+)$"
    match = re.search(pattern, url)
    player_id = match.group(1)
    data["player_id"] = player_id

    try:
        name = soup.select_one('h1[class = "data-header__headline-wrapper"]').text.split("\n")[-1].strip()
    except AttributeError:
        name = None
        return None
    data["name"] = name

    try:
        player_club = soup.select_one("span[class = 'data-header__club']").text.strip()
    except AttributeError:
        player_club = None
        print(f"Không tìm thấy thông tin câu lạc bộ cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        player_club = None
        print(f"Không tìm thấy thông tin câu lạc bộ cho cầu thủ {name}. tại lỗi ValueError")
    except IndexError:
        player_club = None
        print(f"Không tìm thấy thông tin câu lạc bộ cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["player_club"] = player_club

    try:
        age = float(soup.select_one('li[class="data-header__label"]').text.split("\n")[-2].split()[-1].strip("()"))
    except AttributeError:
        age = None
        print(f"Không tìm thấy thông tin tuổi cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        age = None
        print(f"Không tìm thấy thông tin tuổi cho cầu thủ {name}. tại lỗi ValueError")
        return None
    except IndexError:
        age = None
        print(f"Không tìm thấy thông tin tuổi cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["age"] = age

    try:
        position = soup.find('dd', class_='detail-position__position').text
    except AttributeError:
        position = None
        print(f"Không tìm thấy thông tin vị trí cho cầu thủ {name}. tại lỗi AttributeError")
    except ValueError:
        position = None
        print(f"Không tìm thấy thông tin vị trí cho cầu thủ {name}. tại lỗi ValueError")
    except IndexError:
        position = None
        print(f"Không tìm thấy thông tin vị trí cho cầu thủ {name}. tại lỗi IndexError")
    data["position"] = position

    try:
        market_value = soup.select_one('a[class="data-header__market-value-wrapper"]').text.split(" ")[0].replace('€', '')
        if "m" in market_value:
            market_value = market_value.replace("m", "")
            market_value = float(market_value)*1000
        elif "k" in market_value:
            market_value = market_value.replace("k", "")
            market_value = float(market_value)
    except AttributeError:
        market_value = None
        print(f"Không tìm thấy thông tin giá trị thị trường cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        market_value = None
        print(f"Không tìm thấy thông tin giá trị thị trường cho cầu thủ {name}. tại lỗi ValueError")
        return None
    except IndexError:
        market_value = None
        print(f"Không tìm thấy thông tin giá trị thị trường cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["market_value"] = market_value
    
    try:
        all_nationalities = soup.find_all('span', itemprop = "nationality")
        nationality = all_nationalities[0].text.strip()
        nationality = soup.find('span', itemprop = "nationality").text.strip()
    except AttributeError:
        nationality = None
        print(f"Không tìm thấy thông tin quốc tịch cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        nationality = None
        print(f"Không tìm thấy thông tin quốc tịch cho cầu thủ {name}. tại lỗi ValueError")
        return None
    except IndexError:
        nationality = None
        print(f"Không tìm thấy thông tin quốc tịch cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["nationality"] = nationality

    try:
        player_height = float(re.search("Height:.*?([0-9].*?)\n", soup.text, re.DOTALL).group(1).strip().split(" ")[0].replace(",", "."))
    except AttributeError:
        player_height = None
        print(f"Không tìm thấy thông tin chiều cao cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        player_height = None
        print(f"Không tìm thấy thông tin chiều cao cho cầu thủ {name}. tại lỗi ValueError")
        return None
    except IndexError:
        player_height = None
        print(f"Không tìm thấy thông tin chiều cao cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["player_height"] = player_height

    try:
        player_agent = re.search("Agent:.*?([A-z].*?)\n", soup.text, re.DOTALL).group(1).strip()
    except AttributeError:
        player_agent = None
        print(f"Không tìm thấy thông tin người đại diện cho cầu thủ {name}. tại lỗi AttributeError")
        player_agent = "None"
    except ValueError:
        player_agent = None
        print(f"Không tìm thấy thông tin người đại diện cho cầu thủ {name}. tại lỗi ValueError")
        return None
    except IndexError:
        player_agent = None
        print(f"Không tìm thấy thông tin người đại diện cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["player_agent"] = player_agent

    try:
        # name_span = soup.find_all('span', class_='info-table__content info-table__content--bold', string=lambda text:text and text.startswith("Name"))
        
        # if name_span:
        #     strong_foot = soup.select('span[class = "info-table__content info-table__content--bold"]')[6].text
        # else:
        #     strong_foot = soup.select('span[class = "info-table__content info-table__content--bold"]')[5].text
        
        strong_foot = soup.select('span[class = "info-table__content info-table__content--bold"]')[6].text
        name_span = soup.find_all('span', class_='info-table__content info-table__content--regular', string=lambda text:text and 'name' in text.lower())
        if not name_span:
            print(f"Cầu thủ {name} không có thông tin về tên đầy đủ.")
            strong_foot = soup.select('span[class = "info-table__content info-table__content--bold"]')[5].text
    except AttributeError:
        strong_foot = None
        print(f"Không tìm thấy thông tin chân thuận cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        strong_foot = None
        print(f"Không tìm thấy thông tin chân thuận cho cầu thủ {name}. tại lỗi ValueError")
        return None
    except IndexError:
        strong_foot = None
        print(f"Không tìm thấy thông tin chân thuận cho cầu thủ {name}. tại lỗi IndexError")
        return None
    data["strong_foot"] = strong_foot

    try:
        contract_value_time = float(re.search("Contract expires: (.*)", soup.text).group(1).split()[-1])
    except AttributeError:
        contract_value_time = None
        print(f"Không tìm thấy thông tin thời gian hợp đồng cho cầu thủ {name}. tại lỗi AttributeError")
        return None
    except ValueError:
        contract_value_time = "None"
        print(f"Không tìm thấy thông tin thời gian hợp đồng cho cầu thủ {name}. tại lỗi ValueError")
        
    data["contract_value_time"] = contract_value_time

    return data

In [10]:
data_column = ["player_id", "name", "player_club", "age", "position", "market_value", "nationality", "player_height", "player_agent", "strong_foot", "contract_value_time"]

In [11]:
# PATH_TO_PLAYERS_DATA = 'Players_Data.csv'
# players_data = pd.DataFrame(columns = data_column).astype(str)
# hyperlink = 'https://www.transfermarkt.com' + pd.read_csv(PATH_TO_PLAYERS_LINK)

# for i in range(len(hyperlink)):
#     single_player_data = scraping_players_data(hyperlink.loc[i, "0"], players_data)
#     if single_player_data is None:
#         print(f"Error scraping data for player {i+1}: {hyperlink.loc[i, '0']}")
#         single_player_data = scraping_players_data(hyperlink.loc[i, "0"], players_data)
#     if single_player_data is None:
#         print(f"Error scraping data for player {i+1}: {hyperlink.loc[i, '0']}")
#         continue
#     players_data = pd.concat([players_data, pd.DataFrame([single_player_data])], ignore_index=True)
#     #lưu dữ liệu vào file csv
#     if i % 10 == 0:
#         print(f"Scraped {i} players data")
#         players_data.to_csv(PATH_TO_PLAYERS_DATA, index=False)

# pd.DataFrame(players_data).to_csv(PATH_TO_PLAYERS_DATA, index=False)

In [12]:
players_data = pd.read_csv(PATH_TO_PLAYERS_DATA)

not_gk_players_data = players_data[players_data['position'] != 'Goalkeeper']
gk_players_data = players_data[players_data['position'] == 'Goalkeeper']

PATH_TO_NOT_GK_PLAYERS_DATA = 'Not_GK_Players_Data.csv'
PATH_TO_GK_PLAYERS_DATA = 'GK_Players_Data.csv'
not_gk_players_data.to_csv(PATH_TO_NOT_GK_PLAYERS_DATA, index=False)
gk_players_data.to_csv(PATH_TO_GK_PLAYERS_DATA, index=False)

In [13]:
# gk_players_data = pd.read_csv(PATH_TO_GK_PLAYERS_DATA)
# players_link = pd.read_csv(PATH_TO_PLAYERS_LINK)

# gk_ids = gk_players_data['player_id'].astype(str)

# not_gk_players_link = players_link[~players_link.iloc[:, 0].str.split('/').str[-1].isin(gk_ids)]
# gk_players_link = players_link[players_link.iloc[:, 0].str.split('/').str[-1].isin(gk_ids)]

# PATH_TO_NOT_GK_PLAYERS_LINK = 'Not_GK_Players_Link.csv'
# PATH_TO_GK_PLAYERS_LINK = 'GK_Players_Link.csv'

# not_gk_players_link.to_csv(PATH_TO_NOT_GK_PLAYERS_LINK, index=False)
# gk_players_link.to_csv(PATH_TO_GK_PLAYERS_LINK, index=False)
# Đọc dữ liệu từ file CSV
gk_players_data = pd.read_csv(PATH_TO_GK_PLAYERS_DATA)
not_gk_players_data = pd.read_csv(PATH_TO_NOT_GK_PLAYERS_DATA)
players_link = pd.read_csv(PATH_TO_PLAYERS_LINK)

# Lấy danh sách ID của thủ môn và cầu thủ không phải thủ môn
gk_ids = gk_players_data['player_id'].astype(str)
not_gk_ids = not_gk_players_data['player_id'].astype(str)

# Lọc liên kết của thủ môn và tạo bản sao rõ ràng
gk_players_link = players_link[players_link.iloc[:, 0].str.split('/').str[-1].isin(gk_ids)].copy()

# Trích xuất player_id tạm thời để lọc và sắp xếp
gk_players_link.loc[:, 'player_id'] = gk_players_link.iloc[:, 0].str.split('/').str[-1]
valid_gk_ids = gk_players_data['player_id'].astype(str).tolist()
gk_players_link = gk_players_link[gk_players_link['player_id'].isin(valid_gk_ids)]
gk_players_link = gk_players_link.set_index('player_id').loc[gk_ids].reset_index()

# Lọc liên kết của cầu thủ không phải thủ môn và tạo bản sao rõ ràng
not_gk_players_link = players_link[players_link.iloc[:, 0].str.split('/').str[-1].isin(not_gk_ids)].copy()

# Trích xuất player_id tạm thời để lọc và sắp xếp
not_gk_players_link.loc[:, 'player_id'] = not_gk_players_link.iloc[:, 0].str.split('/').str[-1]
valid_not_gk_ids = not_gk_players_data['player_id'].astype(str).tolist()
not_gk_players_link = not_gk_players_link[not_gk_players_link['player_id'].isin(valid_not_gk_ids)]
not_gk_players_link = not_gk_players_link.set_index('player_id').loc[not_gk_ids].reset_index()

# Định nghĩa đường dẫn file CSV
PATH_TO_NOT_GK_PLAYERS_LINK = 'Not_GK_Players_Link.csv'
PATH_TO_GK_PLAYERS_LINK = 'GK_Players_Link.csv'

# Chỉ lưu cột chứa đường dẫn (cột 0) vào file CSV
not_gk_players_link[['0']].to_csv(PATH_TO_NOT_GK_PLAYERS_LINK, index=False)
gk_players_link[['0']].to_csv(PATH_TO_GK_PLAYERS_LINK, index=False)

In [14]:
def scraping_not_gk_stat(url, name):
    headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36", "accept-language": "en-US,en;q=0.9"}

    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.content, "html.parser")
    not_gk_stat = {}

    try:
        appearances = soup.find_all("td", {"class": "zentriert"})[1].text
        if (appearances == "-"):
            appearances = 0
        else:
            appearances = float(appearances)
    except ValueError:
        appearances = None
    except AttributeError:
        appearances = None
    except IndexError:
        appearances = None
    not_gk_stat["appearances"] = appearances

    try:
        PPG = soup.find_all("td", {"class": "zentriert"})[2].text
        if (PPG == "-"):
            PPG = 0
        else:
            PPG = float(PPG)
    except ValueError:
        PPG = None
    except AttributeError:
        PPG = None
    except IndexError:
        PPG = None
    not_gk_stat["PPG"] = PPG 

    try:
        goals = soup.find_all("td", {"class": "zentriert"})[3].text
        if (goals == "-"):
            goals = 0
        else:
            goals = float(goals)
    except ValueError:
        goals = None
    except AttributeError:
        goals = None
    except IndexError:
        goals = None
    not_gk_stat["goals"] = goals

    try:
        assists = soup.find_all("td", {"class": "zentriert"})[4].text
        if (assists == "-"):
            assists = 0
        else:
            assists = float(assists)
    except ValueError:
        assists = None
    except AttributeError:
        assists = None
    except IndexError:
        assists = None
    not_gk_stat["assists"] = assists

    try:
        own_goals = soup.find_all("td", {"class": "zentriert"})[5].text
        if (own_goals == "-"):
            own_goals = 0
        else:
            own_goals = float(own_goals)
    except ValueError:
        own_goals = None
    except AttributeError:
        own_goals = None
    except IndexError:
        own_goals = None
    not_gk_stat["own_goals"] = own_goals 

    try:
        substitutions_on = soup.find_all("td", {"class": "zentriert"})[6].text
        if (substitutions_on == "-"):
            substitutions_on = 0
        else:
            substitutions_on = float(substitutions_on)
    except ValueError:
        substitutions_on = None
    except AttributeError:
        substitutions_on = None
    except IndexError:
        substitutions_on = None
    not_gk_stat["substitutions_on"] = substitutions_on 

    try:
        substitutions_off = soup.find_all("td", {"class": "zentriert"})[7].text
        if (substitutions_off == "-"):
            substitutions_off = 0
        else:
            substitutions_off = float(substitutions_off)
    except ValueError:
        substitutions_off = None
    except AttributeError:
        substitutions_off = None
    except IndexError:
        substitutions_off = None
    not_gk_stat["substitutions_off"] = substitutions_off
    
    try:
        yellow_cards = soup.find_all("td", {"class": "zentriert"})[8].text
        if (yellow_cards == "-"):
            yellow_cards = 0
        else:
            yellow_cards = float(yellow_cards)
    except ValueError:
        yellow_cards = None
    except AttributeError:
        yellow_cards = None
    except IndexError:
        yellow_cards = None
    not_gk_stat["yellow_cards"] = yellow_cards

    try:
        second_yellow_cards = soup.find_all("td", {"class": "zentriert"})[9].text
        if (second_yellow_cards == "-"):
            second_yellow_cards = 0
        else:
            second_yellow_cards = float(second_yellow_cards)
    except ValueError:
        second_yellow_cards = None
    except AttributeError:
        second_yellow_cards = None
    except IndexError:
        second_yellow_cards = None
    not_gk_stat["second_yellow_cards"] = second_yellow_cards

    try:
        red_cards = soup.find_all("td", {"class": "zentriert"})[10].text
        if (red_cards == "-"):
            red_cards = 0
        else:
            red_cards = float(red_cards)
    except ValueError:
        red_cards = None
    except AttributeError:
        red_cards = None
    except IndexError:
        red_cards = None
    not_gk_stat["red_cards"] = red_cards

    try:
        penalty_goals = soup.find_all("td", {"class": "zentriert"})[11].text
        if (penalty_goals == "-"):
            penalty_goals = 0
        else:
            penalty_goals = float(penalty_goals)
    except ValueError:
        penalty_goals = None
    except AttributeError:
        penalty_goals = None
    except IndexError:
        penalty_goals = None
    not_gk_stat["penalty_goals"] = penalty_goals

    try:
        minutes_per_goal = soup.find_all("td", {"class": "rechts"})[1].text.split("'")[0]
        if (minutes_per_goal == "-"):
            minutes_per_goal = 0
        else:
            minutes_per_goal = float(minutes_per_goal)
    except ValueError:
        minutes_per_goal = None
    except AttributeError:
        minutes_per_goal = None
    except IndexError:
        minutes_per_goal = None
    not_gk_stat["minutes_per_goal"] = minutes_per_goal

    try:
        minutes_played = soup.find_all("td", {"class": "rechts"})[2].text.split("'")[0]
        if (minutes_played == "-"):
            minutes_played = 0
        else:
            minutes_played = float(minutes_played)
    except ValueError:
        minutes_played = None
    except AttributeError:
        minutes_played = None
    except IndexError:
        minutes_played = None
    not_gk_stat["minutes_played"] = minutes_played

    return not_gk_stat

In [15]:
not_gk_stat_column = ["appearances", "PPG", "goals", "assists", "own_goals", "substitutions_on", "substitutions_off", "yellow_cards", "second_yellow_cards", "red_cards", "penalty_goals", "minutes_per_goal", "minutes_played"]

In [16]:
# not_gk_players_stat = pd.DataFrame(columns = not_gk_stat_column).astype(str)
# not_gk_players_link = 'https://www.transfermarkt.com' + pd.read_csv(PATH_TO_NOT_GK_PLAYERS_LINK)

# for i in range(len(not_gk_players_link)):
#     if i%10 == 0:
#         print(i, end=" ")
#         #Lưu dữ liệu vào file csv
#         pd.DataFrame(not_gk_players_stat).to_csv(PATH_TO_NOT_GK_PLAYERS_STAT, index=False)

#     id = not_gk_players_link.loc[i, "0"].split('spieler/')[-1]
#     name = not_gk_players_link.loc[i, "0"].split('com/')[-1].split('/profil')[0].replace(' ', '-')
#     not_gk_players_hyperlink = f"https://www.transfermarkt.com/{name}/leistungsdatendetails/spieler/{id}/plus/1?saison=2024&verein=&liga=&wettbewerb=&pos=&trainer_id="
#     single_not_gk_player_stat = scraping_not_gk_stat(not_gk_players_hyperlink, name)
#     not_gk_players_stat = pd.concat([not_gk_players_stat, pd.DataFrame([single_not_gk_player_stat])], ignore_index=True)

# PATH_TO_NOT_GK_PLAYERS_STAT = 'Not_GK_Players_Stat.csv'

# pd.DataFrame(not_gk_players_stat).to_csv(PATH_TO_NOT_GK_PLAYERS_STAT, index=False)
import os
if os.path.exists(PATH_TO_NOT_GK_PLAYERS_STAT):
    not_gk_players_stat = pd.read_csv(PATH_TO_NOT_GK_PLAYERS_STAT)  # Đọc dữ liệu cũ nếu file tồn tại
else:
    not_gk_players_stat = pd.DataFrame(columns=not_gk_stat_column).astype(str)  # Tạo mới nếu không tồn tại

# Đọc file CSV chứa link
not_gk_players_link = pd.read_csv(PATH_TO_NOT_GK_PLAYERS_LINK)
not_gk_players_link['0'] = 'https://www.transfermarkt.com' + not_gk_players_link['0'].astype(str)

# Đếm số đối tượng đã có trong DataFrame
count = len(not_gk_players_stat)
print(f"Số lượng đối tượng đã có trong DataFrame: {count}")

# Vòng lặp xử lý
for i in range(count, len(not_gk_players_link)): #3120
    if i % 10 == 0:
        print(i, end=" ")
        # Ghi dữ liệu vào file CSV (thêm mà không ghi header)
        not_gk_players_stat.to_csv(PATH_TO_NOT_GK_PLAYERS_STAT, index=False)

    id = not_gk_players_link.loc[i, "0"].split('spieler/')[-1]
    name = not_gk_players_link.loc[i, "0"].split('com/')[-1].split('/profil')[0].replace('-', ' ')
    not_gk_players_hyperlink = f"https://www.transfermarkt.com/{name}/leistungsdatendetails/spieler/{id}/plus/1?saison=2024&verein=&liga=&wettbewerb=&pos=&trainer_id="
    single_not_gk_player_stat = scraping_not_gk_stat(not_gk_players_hyperlink, name)
    not_gk_players_stat = pd.concat([not_gk_players_stat, pd.DataFrame([single_not_gk_player_stat])], ignore_index=True)

pd.DataFrame(not_gk_players_stat).to_csv(PATH_TO_NOT_GK_PLAYERS_STAT, index=False)

Số lượng đối tượng đã có trong DataFrame: 4440
4440 4450 

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15112\3631455229.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  not_gk_players_stat = pd.concat([not_gk_players_stat, pd.DataFrame([single_not_gk_player_stat])], ignore_index=True)


4460 4470 4480 4490 4500 4510 4520 4530 

In [17]:
def scraping_gk_stat(url, name):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
               "accept-language": "en-US,en;q=0.9"}
    page = requests.get(url, headers=headers)
    soup = BeautifulSoup(page.content, "html.parser")
    gk_stat = {}

    try:
        appearances = soup.find_all("td", {"class": "zentriert"})[1].text
        if (appearances == "-"):
            appearances = 0
        else:
            appearances = float(appearances)
    except ValueError:
        appearances = None
    except AttributeError:
        appearances = None
    except IndexError:
        appearances = None
    gk_stat["appearances"] = appearances

    try:
        PPG = soup.find_all("td", {"class": "zentriert"})[2].text
        if (PPG == "-"):
            PPG = 0
        else:
            PPG = float(PPG)
    except ValueError:
        PPG = None
    except AttributeError:
        PPG = None
    except IndexError:
        PPG = None
    gk_stat["PPG"] = PPG 

    try:
        goals = soup.find_all("td", {"class": "zentriert"})[3].text
        if (goals == "-"):
            goals = 0
        else:
            goals = float(goals)
    except ValueError:
        goals = None
    except AttributeError:
        goals = None
    except IndexError:
        goals = None
    gk_stat["goals"] = goals

    try:
        own_goals = soup.find_all("td", {"class": "zentriert"})[4].text
        if (own_goals == "-"):
            own_goals = 0
        else:
            own_goals = float(own_goals)
    except ValueError:
        own_goals = None
    except AttributeError:
        own_goals = None
    except IndexError:
        own_goals = None
    gk_stat["own_goals"] = own_goals 

    try:
        substitutions_on = soup.find_all("td", {"class": "zentriert"})[5].text
        if (substitutions_on == "-"):
            substitutions_on = 0
        else:
            substitutions_on = float(substitutions_on)
    except ValueError:
        substitutions_on = None
    except AttributeError:
        substitutions_on = None
    except IndexError:
        substitutions_on = None
    gk_stat["substitutions_on"] = substitutions_on 

    try:
        substitutions_off = soup.find_all("td", {"class": "zentriert"})[6].text
        if (substitutions_off == "-"):
            substitutions_off = 0
        else:
            substitutions_off = float(substitutions_off)
    except ValueError:
        substitutions_off = None
    except AttributeError:
        substitutions_off = None
    except IndexError:
        substitutions_off = None
    gk_stat["substitutions_off"] = substitutions_off
    
    try:
        yellow_cards = soup.find_all("td", {"class": "zentriert"})[7].text
        if (yellow_cards == "-"):
            yellow_cards = 0
        else:
            yellow_cards = float(yellow_cards)
    except ValueError:
        yellow_cards = None
    except AttributeError:
        yellow_cards = None
    except IndexError:
        yellow_cards = None
    gk_stat["yellow_cards"] = yellow_cards

    try:
        second_yellow_cards = soup.find_all("td", {"class": "zentriert"})[8].text
        if (second_yellow_cards == "-"):
            second_yellow_cards = 0
        else:
            second_yellow_cards = float(second_yellow_cards)
    except ValueError:
        second_yellow_cards = None
    except AttributeError:
        second_yellow_cards = None
    except IndexError:
        second_yellow_cards = None
    gk_stat["second_yellow_cards"] = second_yellow_cards

    try:
        red_cards = soup.find_all("td", {"class": "zentriert"})[9].text
        if (red_cards == "-"):
            red_cards = 0
        else:
            red_cards = float(red_cards)
    except ValueError:
        red_cards = None
    except AttributeError:
        red_cards = None
    except IndexError:
        red_cards = None
    gk_stat["red_cards"] = red_cards

    try:
        goals_conceded = soup.find_all("td", {"class": "zentriert"})[10].text
        if (goals_conceded == "-"):
            goals_conceded = 0
        else:
            goals_conceded = float(goals_conceded)
    except ValueError:
        goals_conceded = None
    except AttributeError:
        goals_conceded = None
    except IndexError:
        goals_conceded = None
    gk_stat["goals_conceded"] = goals_conceded

    try:
        clean_sheet = soup.find_all("td", {"class": "zentriert"})[11].text
        if (clean_sheet == "-"):
            clean_sheet = 0
        else:
            clean_sheet = float(clean_sheet)
    except ValueError:
        clean_sheet = None
    except AttributeError:
        clean_sheet = None
    except IndexError:
        clean_sheet = None
    gk_stat["clean_sheet"] = clean_sheet

    try:
        minutes_played = soup.find_all("td", {"class": "rechts"})[1].text.split("'")[0]
        if (minutes_played == "-"):
            minutes_played = 0
        else:
            minutes_played = float(minutes_played)
    except ValueError:
        minutes_played = None
    except AttributeError:
        minutes_played = None
    except IndexError:
        minutes_played = None
    gk_stat["minutes_played"] = minutes_played

    return gk_stat

In [18]:
gk_stat_column = ["appearances", "PPG", "goals", "own_goals", "substitutions_on", "substitutions_off", "yellow_cards", "second_yellow_cards", "red_cards", "goals_conceded", "clean_sheet", "minutes_played"]

In [20]:
gk_players_stat = pd.DataFrame(columns = gk_stat_column).astype(str)
gk_players_link = 'https://www.transfermarkt.com' + pd.read_csv(PATH_TO_NOT_GK_PLAYERS_LINK)

for i in range(len(gk_players_link)):
    if i%10 == 0:
        print(i, end=" ")
        #Lưu dữ liệu vào file csv
        pd.DataFrame(gk_players_stat).to_csv(PATH_TO_GK_PLAYERS_STAT, index=False)
    id = gk_players_link.loc[i, "0"].split('spieler/')[-1]
    name = gk_players_link.loc[i, "0"].split('com/')[-1].split('/profil')[0].replace(' ', '-')
    gk_players_hyperlink = f"https://www.transfermarkt.com/{name}/leistungsdatendetails/spieler/{id}/plus/1?saison=2024&verein=&liga=&wettbewerb=&pos=&trainer_id="
    single_gk_player_stat = scraping_gk_stat(gk_players_hyperlink, name)
    gk_players_stat = pd.concat([gk_players_stat, pd.DataFrame([single_gk_player_stat])], ignore_index=True)

PATH_TO_GK_PLAYERS_STAT = 'GK_Players_Stat.csv'
pd.DataFrame(gk_players_stat).to_csv(PATH_TO_GK_PLAYERS_STAT, index=False)

0 

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15112\666869513.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gk_players_stat = pd.concat([gk_players_stat, pd.DataFrame([single_gk_player_stat])], ignore_index=True)


10 20 30 40 50 60 70 

KeyboardInterrupt: 

In [ ]:
# not_gk_players_data = pd.read_csv(PATH_TO_NOT_GK_PLAYERS_DATA)
# gk_players_data = pd.read_csv(PATH_TO_GK_PLAYERS_DATA)
# not_gk_players_stat = pd.read_csv(PATH_TO_NOT_GK_PLAYERS_STAT)
# gk_players_stat = pd.read_csv(PATH_TO_GK_PLAYERS_STAT)

In [ ]:
# not_gk_players_data['Index'] = range(1, len(not_gk_players_data) + 1)
# cols = ['Index'] + [col for col in not_gk_players_data.columns if col != 'Index']
# not_gk_players_data.to_csv(PATH_TO_NOT_GK_PLAYERS_DATA, index=False)

# gk_players_data['Index'] = range(1, len(gk_players_data) + 1)
# cols = ['Index'] + [col for col in gk_players_data.columns if col != 'Index']
# gk_players_data.to_csv(PATH_TO_GK_PLAYERS_DATA, index=False)

# not_gk_players_stat['Index'] = range(1, len(not_gk_players_stat) + 1)
# cols = ['Index'] + [col for col in not_gk_players_stat.columns if col != 'Index']
# not_gk_players_stat.to_csv(PATH_TO_NOT_GK_PLAYERS_STAT, index=False)

# gk_players_stat['Index'] = range(1, len(gk_players_stat) + 1)
# cols = ['Index'] + [col for col in gk_players_stat.columns if col != 'Index']
# gk_players_stat.to_csv(PATH_TO_GK_PLAYERS_STAT, index=False)

In [ ]:
# not_gk_players = pd.merge(not_gk_players_data, not_gk_players_stat, on='Index')
# gk_players = pd.merge(gk_players_data, gk_players_stat, on='Index')

# PATH_TO_NOT_GK_PLAYERS = 'Not_GK_Players.csv'
# PATH_TO_GK_PLAYERS = 'GK_Players.csv'
# not_gk_players.to_csv(PATH_TO_NOT_GK_PLAYERS, index=False)
# gk_players.to_csv(PATH_TO_GK_PLAYERS, index=False)

In [ ]:
# not_gk_players = pd.read_csv(PATH_TO_NOT_GK_PLAYERS)
# gk_players = pd.read_csv(PATH_TO_GK_PLAYERS)

In [ ]:
# not_gk_players['goalkeeper_or_not'] = not_gk_players['position'].apply(lambda x: '1' if x == 'Goalkeeper' else '0')
# not_gk_players['goals_conceded'] = float(0)
# not_gk_players['clean_sheet'] = float(0)
# cols = list(not_gk_players.columns)
# position_index_4 = cols.index('position')
# position_index_5 = cols.index('red_cards')
# cols.insert(position_index_4 + 1, cols.pop(cols.index('goalkeeper_or_not')))
# cols.insert(position_index_5 + 1, cols.pop(cols.index('goals_conceded')))
# cols.insert(position_index_5 + 2, cols.pop(cols.index('clean_sheet')))
# not_gk_players = not_gk_players[cols]
# not_gk_players.head(1)
# not_gk_players.to_csv(PATH_TO_NOT_GK_PLAYERS)
# not_gk_players = pd.read_csv(PATH_TO_NOT_GK_PLAYERS)

In [ ]:
# gk_players['goalkeeper_or_not'] = gk_players['position'].apply(lambda x: '1' if x == 'Goalkeeper' else '0')
# gk_players['assists'] = float(0)
# gk_players['penalty_goals'] = float(0)
# gk_players['minutes_per_goal'] = float(0)
# cols = list(gk_players.columns)
# position_index_1 = cols.index('position')
# position_index_2 = cols.index('goals')
# position_index_3 = cols.index('clean_sheet')
# cols.insert(position_index_1 + 1, cols.pop(cols.index('goalkeeper_or_not')))
# cols.insert(position_index_2 + 1, cols.pop(cols.index('assists')))
# cols.insert(position_index_3 + 1, cols.pop(cols.index('penalty_goals')))
# cols.insert(position_index_3 + 2, cols.pop(cols.index('minutes_per_goal')))
# gk_players = gk_players[cols]
# gk_players.head(1)
# gk_players.to_csv(PATH_TO_GK_PLAYERS)
# gk_players = pd.read_csv(PATH_TO_GK_PLAYERS)

In [ ]:
# final_data = pd.concat([not_gk_players, gk_players])

In [ ]:
# PATH_TO_FINAL_DATA = 'Final_Data.csv'
# final_data.to_csv(PATH_TO_FINAL_DATA, index=False)
# print("Data scraping completed and saved to Final_Data.csv")

In [ ]:
# import json
# import pandas as pd
# from kafka import KafkaProducer
# from kafka.admin import KafkaAdminClient, NewTopic
# from kafka.errors import KafkaError
# import logging
# import time
# from datetime import datetime

# # Cấu hình logging
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(levelname)s - %(message)s',
#     handlers=[
#         logging.StreamHandler()  # In log ra console
#     ]
# )
# logger = logging.getLogger(__name__)

# # Kafka configuration
# KAFKA_BROKER_URL = 'localhost:9092'
# KAFKA_TOPIC = 'football_players'

# # Kiểm tra và tạo topic nếu chưa tồn tại
# def check_and_create_topic():
#     try:
#         admin_client = KafkaAdminClient(bootstrap_servers=[KAFKA_BROKER_URL])
#         topic_list = admin_client.list_topics()
#         if KAFKA_TOPIC not in topic_list:
#             logger.info(f"Topic '{KAFKA_TOPIC}' does not exist. Creating...")
#             new_topic = NewTopic(name=KAFKA_TOPIC, num_partitions=1, replication_factor=1)
#             admin_client.create_topics(new_topics=[new_topic], validate_only=False)
#             logger.info(f"Topic '{KAFKA_TOPIC}' created successfully.")
#         else:
#             logger.info(f"Topic '{KAFKA_TOPIC}' already exists.")
#         admin_client.close()
#     except Exception as e:
#         logger.error(f"Error while checking/creating topic: {e}")
#         raise

# # Gửi dữ liệu từ DataFrame vào Kafka
# def send_to_kafka(producer, df, topic):
#     logger.info(f"Starting to send {len(df)} records to Kafka topic '{topic}'...")
#     for index, row in df.iterrows():
#         # Chuyển dòng thành dictionary và loại bỏ các giá trị NaN
#         record = row.to_dict()
#         for key, value in record.items():
#             if pd.isna(value):
#                 record[key] = None
        
#         # Chuyển dictionary thành JSON string
#         record_json = json.dumps(record, ensure_ascii=False)
        
#         # Gửi message vào Kafka
#         try:
#             producer.send(topic, value=record_json.encode('utf-8'))
#             logger.info(f"Message sent to {topic} for record {index + 1}")
#         except KafkaError as e:
#             logger.error(f"Failed to send message for record {index + 1}: {e}")
        
#         # Đảm bảo message được gửi đi (flush sau mỗi 100 bản ghi)
#         if index % 100 == 0:
#             producer.flush()
#             logger.info(f"Sent {index + 1} records so far...")
        
#         # Thêm delay nhỏ để tránh gửi quá nhanh
#         time.sleep(0.1)

#     # Flush lần cuối để đảm bảo tất cả message được gửi
#     producer.flush()
#     logger.info("Finished sending all data to Kafka.")

# # Đọc final_data từ file CSV và gửi vào Kafka
# def main():
#     # Đường dẫn file final_data
#     final_data_file = 'Final_Data.csv'
    
#     # Đọc dữ liệu từ file
#     try:
#         final_data = pd.read_csv(final_data_file)
#         logger.info(f"Successfully loaded data from {final_data_file}. Total records: {len(final_data)}")
#     except Exception as e:
#         logger.error(f"Failed to load data from {final_data_file}: {e}")
#         raise
    
#     # Kiểm tra và tạo topic
#     try:
#         check_and_create_topic()
#     except Exception as e:
#         logger.error(f"Error while checking/creating topic: {e}")
#         raise
    
#     # Khởi tạo Kafka Producer
#     try:
#         logger.info(f"Connecting to Kafka broker at {KAFKA_BROKER_URL}...")
#         producer = KafkaProducer(
#             bootstrap_servers=[KAFKA_BROKER_URL],
#             value_serializer=lambda v: v,  # Đã encode trước khi gửi
#             retries=5,
#             acks='all'
#         )
#         logger.info("Successfully connected to Kafka.")
#     except Exception as e:
#         logger.error(f"Error connecting to Kafka: {e}")
#         raise
    
#     # Gửi dữ liệu vào Kafka
#     try:
#         send_to_kafka(producer, final_data, KAFKA_TOPIC)
#         logger.info("All data sent to Kafka successfully!")
#     except Exception as e:
#         logger.error(f"Error while sending data to Kafka: {e}")
#         raise
#     finally:
#         logger.info("Closing Kafka producer.")
#         producer.flush(timeout=60)
#         producer.close()

# if __name__ == "__main__":
#     logger.info("Starting Kafka producer script...")
#     main()
#     logger.info("Kafka producer script completed.")

In [ ]:
#All Path used in thi s code
PATH_TO_PLAYERS_LINK = "All_Players_Link.csv"
PATH_TO_PLAYERS_DATA = "Players_Data.csv"
PATH_TO_NOT_GK_PLAYERS_DATA = "Not_GK_Players_Data.csv"
PATH_TO_GK_PLAYERS_DATA = "GK_Players_Data.csv"
PATH_TO_NOT_GK_PLAYERS_LINK = "Not_GK_Players_Link.csv"
PATH_TO_GK_PLAYERS_LINK = "GK_Players_Link.csv"
PATH_TO_NOT_GK_PLAYERS_STAT = "Not_GK_Players_Stat.csv"
PATH_TO_GK_PLAYERS_STAT = "GK_Players_Stat.csv"
PATH_TO_NOT_GK_PLAYERS = 'Not_GK_Players.csv'
PATH_TO_GK_PLAYERS = 'GK_Players_Data.csv'